## ECCO-TCP Passage Metadata and Topic Modeling

### Overview¶

This notebook builds and runs a vector-based semantic search engine for the ECCO-TCP corpus. Steps:

1. Install the necessary libraries.
2. Load the 500-token passages from the `complete_ecco_tcp_passages` subfolder.
3. Generate vector embeddings for each passage using a Sentence-Transformer model.
4. Build a FAISS index for efficient similarity searching.
5. Save the generated model files (embeddings and index) to your project folder to avoid re-computing them every time.
6. Launch an interactive search loop that allows for Rocchio-based relevance feedback to refine search results.

### Cell 2: Install Required Libraries

In [ ]:
%pip install sentence-transformers faiss-cpu numpy

### Cell 3: Imports and Configuration

In [ ]:
import os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import pickle

print("Libraries imported successfully.")

# 1. Set the path to the directory containing the 500-token passages.
#    The notebook is in 'ecco_tcp_standardized_v2', and the data is in a subfolder.
CORPUS_DIR = './complete_ecco_tcp_passages/'

# 2. Define the pre-trained model.
MODEL_NAME = 'all-MiniLM-L6-v2'

# 3. Define the file paths for the saved model components.
#    These will be saved in the main 'ecco_tcp_standardized_v2' folder.
EMBEDDINGS_FILE = 'ecco_v2_embeddings.npy'
PASSAGES_FILE = 'ecco_v2_passages.pkl'
INDEX_FILE = 'ecco_v2_faiss.index'

print(f"Configuration set. Corpus directory: '{CORPUS_DIR}'")

### Cell 4: Core Functions for Building the Model

In [ ]:
def load_passages(directory):
    """Reads all .txt files from a directory and stores their content and filenames."""
    passages = []
    filenames = []
    print(f"Loading passages from: {directory}")
    if not os.path.exists(directory):
        print(f"Error: Directory '{directory}' not found.")
        return None, None
        
    for filename in sorted(os.listdir(directory)):
        if filename.endswith('.txt'):
            filepath = os.path.join(directory, filename)
            try:
                with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                    passages.append(f.read())
                    filenames.append(filename)
            except Exception as e:
                print(f"Error reading {filename}: {e}")
    
    print(f"Found and loaded {len(passages)} passages.")
    return passages, filenames

def generate_embeddings(passages, model_name):
    """
    Generates vector embeddings for a list of text passages, ensuring the
    maximum sequence length is adequate to capture the full 500-word passages.
    """
    print(f"Loading sentence-transformer model: '{model_name}'...")
    # Set max_seq_length to the model's maximum, overriding the library's shorter default of 256 tokens.
    max_len = 512
    model = SentenceTransformer(model_name)
    model.max_seq_length = max_len
    
    print(f"Model sequence length set to {model.max_seq_length} tokens.")
    
    print("Generating embeddings for all passages...")
    embeddings = model.encode(passages, show_progress_bar=True)
    print(f"Embeddings generated with shape: {embeddings.shape}")
    return embeddings

def create_faiss_index(embeddings):
    """Creates a FAISS index from a set of embeddings."""
    d = embeddings.shape[1]
    print(f"Creating a FAISS index with dimension {d}.")
    index = faiss.IndexFlatL2(d)
    index.add(embeddings)
    print(f"FAISS index created. Total vectors in index: {index.ntotal}")
    return index

print("Core functions defined.")

### Cell 5: Build the Model or Load from File

In [ ]:
# Check if the processed files already exist.
if os.path.exists(EMBEDDINGS_FILE) and os.path.exists(PASSAGES_FILE) and os.path.exists(INDEX_FILE):
    print("Loading pre-processed model from files...")
    all_passages, all_filenames = pickle.load(open(PASSAGES_FILE, 'rb'))
    embeddings = np.load(EMBEDDINGS_FILE)
    index = faiss.read_index(INDEX_FILE)
    
    # We still need to load the model for encoding new queries
    model = SentenceTransformer(MODEL_NAME)
    model.max_seq_length = 512 # Ensure query encoding also uses the correct length
    
    print("Model loaded successfully.")
else:
    print("No pre-processed model found. Building from scratch with CORRECTED sequence length...")
    # Step 1: Load the text data.
    all_passages, all_filenames = load_passages(CORPUS_DIR)

    if all_passages:
        # Step 2: Generate embeddings using our FIXED function.
        embeddings = generate_embeddings(all_passages, MODEL_NAME)
        
        # Step 3: Create the FAISS index.
        index = create_faiss_index(embeddings)
        
        # Step 4: Save everything for future use.
        print("Saving processed model components to files...")
        np.save(EMBEDDINGS_FILE, embeddings)
        with open(PASSAGES_FILE, 'wb') as f:
            pickle.dump((all_passages, all_filenames), f)
        faiss.write_index(index, INDEX_FILE)
        
        # Also load the model for searching.
        model = SentenceTransformer(MODEL_NAME)
        model.max_seq_length = 512 # Ensure query encoding also uses the correct length

        print("Model building and saving complete.")

### Cell 6: Search and Relevance Feedback Functions

This version correctly configures the model to use a 512-token context window.

In [ ]:
def search(index, passages, filenames, query_vector=None, query_text=None, model=None, top_k=5):
    """Performs a search using either a raw text query or a pre-computed vector."""
    if query_vector is None:
        if query_text and model:
            print(f"\nEncoding and searching for: '{query_text}'")
            query_vector = model.encode([query_text])
        else:
            raise ValueError("Must provide either a query_vector or both query_text and a model.")
    
    distances, indices = index.search(query_vector, top_k)
    
    print(f"\nTop {top_k} results:")
    for i, idx in enumerate(indices[0]):
        print(f"--- Result [{i}] (Distance: {distances[0][i]:.4f}) ---")
        print(f"Source File: {filenames[idx]}")
        print(f"Passage: {passages[idx][:300]}...")
        print("-" * 20)
        
    return indices[0], distances[0]

def perform_relevance_feedback(original_query_vector, relevant_indices, irrelevant_indices, all_embeddings, alpha=1.0, beta=0.75, gamma=0.15):
    """Calculates a new query vector using the Rocchio algorithm."""
    print("\nUpdating query vector with relevance feedback...")
    relevant_vectors = all_embeddings[relevant_indices]
    irrelevant_vectors = all_embeddings[irrelevant_indices]
    
    centroid_relevant = np.mean(relevant_vectors, axis=0) if len(relevant_vectors) > 0 else np.zeros(original_query_vector.shape)
    centroid_irrelevant = np.mean(irrelevant_vectors, axis=0) if len(irrelevant_vectors) > 0 else np.zeros(original_query_vector.shape)

    new_query_vector = (alpha * original_query_vector) + (beta * centroid_relevant) - (gamma * centroid_irrelevant)
    return new_query_vector

print("Search and feedback functions defined.")

### Cell 7: Launch Interactive Search

#### Make sure to specify the query
NB that the query can be entered in the input box, once this cell is run.

In [ ]:
if 'index' not in locals() or 'model' not in locals():
    print("Model not loaded. Please run the 'Build or Load' cell (Cell 5) before running this cell.")
else:
    # Start the interactive search loop
    initial_query = input("sublime, sublimity, terror, awe, astonishment, vast, infinite, obscurity, Burke, taste, genius, imagination, ruins, mountains, tempest, thunder, horror, wonder")
    
    query_vector = model.encode([initial_query])
    
    while True:
        result_indices, result_distances = search(index, all_passages, all_filenames, query_vector=query_vector, top_k=5)
        
        print("\nProvide feedback by entering the numbers of the results.")
        print("Example: relevant '0 2', irrelevant '3 4'. Press Enter to quit.")
        
        rel_input = input("Enter RELEVANT result numbers, separated by spaces: ")
        if not rel_input.strip():
            print("Exiting search loop.")
            break
            
        irrel_input = input("Enter IRRELEVANT result numbers, separated by spaces: ")
        
        try:
            # Get the actual document indices from the search results
            relevant_feedback_indices = [result_indices[int(i)] for i in rel_input.split()]
            irrelevant_feedback_indices = [result_indices[int(i)] for i in irrel_input.split()]

            # Update the query vector using the feedback
            query_vector = perform_relevance_feedback(
                original_query_vector=query_vector,
                relevant_indices=relevant_feedback_indices,
                irrelevant_indices=irrelevant_feedback_indices,
                all_embeddings=embeddings
            )
        except (ValueError, IndexError) as e:
            print(f"Invalid input. Please enter numbers from 0 to 4. Error: {e}")
            print("Restarting search with the last valid query...")

### Cell 8: Download csv of top results

In [ ]:
import csv

# --- Step 1: Configuration ---

FEEDBACK_K = 8         # Number of results to show during the feedback loop.
DOWNLOAD_K = 100       # Number of results to download to the CSV file.
OUTPUT_CSV_FILENAME = 'refined_search_results.csv'

# --- Step 2: Ensure the model is loaded ---

if 'model' not in locals() or 'index' not in locals():
    print("Model components not found in memory.")
    print("Please ensure you have run Cell 5 to build or load the model before running this cell.")
else:
    # --- PHASE 1: INTERACTIVE REFINEMENT LOOP ---
    
    print("--- Starting Interactive Refinement Phase ---")
    initial_query = input("Enter your initial search query: ")
    
    # Encode the initial query
    query_vector = model.encode([initial_query])
    user_wants_to_download = False
    
    while True:
        # Perform a search for a small number of results for feedback
        print(f"\n--- Displaying Top {FEEDBACK_K} Results for Feedback ---")
        result_indices, _ = search(index, all_passages, all_filenames, query_vector=query_vector, top_k=FEEDBACK_K)
        
        # Ask the user what to do next
        print("\nWhat would you like to do next?")
        choice = input("[F]eedback to refine, [D]ownload results, or [Q]uit: ").lower().strip()
        
        if choice == 'f':
            # --- Perform a round of relevance feedback ---
            try:
                rel_input = input("Enter RELEVANT result numbers (e.g., '0 2'): ")
                irrel_input = input("Enter IRRELEVANT result numbers (e.g., '3'): ")
                
                relevant_feedback_indices = [result_indices[int(i)] for i in rel_input.split()]
                irrelevant_feedback_indices = [result_indices[int(i)] for i in irrel_input.split()]

                # Update the query vector using the feedback
                query_vector = perform_relevance_feedback(
                    original_query_vector=query_vector,
                    relevant_indices=relevant_feedback_indices,
                    irrelevant_indices=irrelevant_feedback_indices,
                    all_embeddings=embeddings
                )
                continue # Continue to the next loop iteration to show new results
                
            except (ValueError, IndexError) as e:
                print(f"Invalid input. Please enter numbers from 0 to {FEEDBACK_K-1}. Error: {e}")
                print("Restarting loop with the last valid query...")
        
        elif choice == 'd':
            # --- Exit the loop to proceed to download ---
            user_wants_to_download = True
            break
            
        elif choice == 'q':
            # --- Quit the entire process ---
            print("Quitting process.")
            break
            
        else:
            print("Invalid choice. Please enter 'f', 'd', or 'q'.")

    # --- PHASE 2: FINAL SEARCH AND DOWNLOAD ---
    
    if user_wants_to_download:
        print(f"\n--- Performing final search for top {DOWNLOAD_K} results... ---")
        
        # Use the final, refined query_vector to perform the large search
        distances, indices = index.search(query_vector, DOWNLOAD_K)
        
        # Prepare the data for the CSV
        results_data = [['Rank', 'Distance', 'Filename']]
        for i in range(DOWNLOAD_K):
            doc_index = indices[0][i]
            rank = i + 1
            distance = distances[0][i]
            filename = all_filenames[doc_index]
            results_data.append([rank, distance, filename])
            
        # Write the data to the CSV file
        try:
            with open(OUTPUT_CSV_FILENAME, 'w', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerows(results_data)
                
            print(f"\nSuccess! The top {DOWNLOAD_K} refined results have been saved to '{OUTPUT_CSV_FILENAME}'.")
            
        except Exception as e:
            print(f"\nAn error occurred while writing the file: {e}")

### Cell 9: Join search results with metadata

In [ ]:
%pip install pandas

In [ ]:
import pandas as pd
import os

# --- 1. Configuration ---

# Define the filenames for your input and output files.
search_results_file = 'search_results_initial_principle_run.csv'
metadata_file = 'ECCOTCP.csv'
output_file = 'search_results_with_metadata.csv'

# --- 2. Load the CSV files into pandas DataFrames ---

try:
    print(f"Loading search results from '{search_results_file}'...")
    search_df = pd.read_csv(search_results_file)
    
    print(f"Loading metadata from '{metadata_file}'...")
    meta_df = pd.read_csv(metadata_file)
    
    print("Files loaded successfully.")
    
except FileNotFoundError as e:
    print(f"\nERROR: Could not find a required file: {e}")
    print("Please make sure both CSV files are in the same directory as your notebook.")

else:
    # --- 3. Prepare the DataFrames for Merging ---
    
    # In the search results DataFrame, create a new 'JoinKey' column.
    # This key will be the first part of the 'Filename' string, which matches the 'TCP' ID.
    # For example, 'K022135.000.00002.txt' becomes 'K022135.000'.
    # We use .str[:11] because 'K022135.000' is 11 characters long.
    print("Creating join key...")
    search_df['JoinKey'] = search_df['Filename'].str[:11]
    
    # --- 4. Perform the Merge ---
    
    print("Merging dataframes...")
    # We will perform a 'left' merge.
    # This keeps every row from the search_df (our "left" dataframe).
    # It then adds columns from meta_df wherever the 'JoinKey' in search_df
    # matches the 'TCP' column in meta_df.
    merged_df = pd.merge(
        left=search_df,
        right=meta_df,
        left_on='JoinKey',
        right_on='TCP',
        how='left'  # 'left' ensures we keep all our original search results
    )
    
    # --- 5. Clean Up the Final DataFrame ---
    
    # We no longer need the temporary 'JoinKey' or the redundant 'TCP' column.
    merged_df = merged_df.drop(columns=['JoinKey', 'TCP'])
    
    # You can also reorder the columns for better readability if you like.
    # Let's bring the new metadata columns to the front.
    # First, get the columns you want at the start.
    desired_order = ['Rank', 'Distance', 'Filename', 'Author', 'Title', 'Date', 'Pages']
    # Get the remaining columns from the original search results (if any).
    remaining_cols = [col for col in search_df.columns if col not in desired_order and col != 'JoinKey']
    
    # Combine the lists to create the final column order.
    final_column_order = desired_order + remaining_cols
    
    # Apply the new order.
    merged_df = merged_df[final_column_order]
    
    # --- 6. Save the Result to a New CSV File ---
    
    try:
        # index=False prevents pandas from writing its internal row numbers to the file.
        merged_df.to_csv(output_file, index=False)
        print(f"\nSuccess! The enriched data has been saved to '{output_file}'.")
        
        # Display the first few rows of the new, merged table right here in the notebook.
        print("\nHere's a preview of your new data:")
        display(merged_df.head())
        
    except Exception as e:
        print(f"\nAn error occurred while saving the file: {e}")